# Full query set (100%): model comparison table

Same layout as `budget-reduction-table.ipynb`, restricted to the runs that keep every query
(100%). At 100% the random and shared template rules keep the same queries, so there is one table.

Each experiment gets an extra column, **Greatest difference between models**: the absolute Macro-F1 gap
between the best and the worst of smol, qwen and granite (the largest pairwise absolute
difference), in percentage points. It is one value per cache setting and dataset, shared by the
three model rows, and is shaded on a light-orange scale from 0% (white) to 100% (full orange).

**Topic columns use the final experiments** (`white-box-cat-final`, `black-box-cat-final`):
the revised, leak-free per-noun experiments with 500 trees per ensemble, matching the
frequency experiments.

Outputs in `full_queries_tables/`:
- `full_queries_table.csv`: the table, with the difference repeated on each model row.
- `full_queries_table.tex`: LaTeX version (needs `booktabs`, `multirow` and `xcolor` with the
  `table` option).


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(".")
OUTPUT_DIRECTORY = Path("full_queries_tables")
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

EXPERIMENTS = (
    # (label, folder, task, box)
    ("White-box. Content frequency", "white-box-freq", "freq", "white"),
    ("Black-box. Content frequency", "black-box-freq", "freq", "black"),
    # Topic results come from the final experiments (revised method, 500 trees);
    # see classificators/final_topic.py.
    ("White-box. Topic of use", "white-box-cat-final", "topic", "white"),
    ("Black-box. Topic of use", "black-box-cat-final", "topic", "black"),
)
DATASETS = ("pg19", "pubmed")
DATASET_LABELS = {"pg19": "PG19", "pubmed": "PubMed"}
CACHE_SETTINGS = {"no_cache": False, "cache": True}
CACHE_LABELS = {False: "No cache", True: "Cache"}
MODELS = ("smol", "qwen", "granite")
DIFFERENCE = "Greatest difference between models"

## 1. Collect the 100% runs

In [2]:
frames = []
missing = []
for label, folder, task, box in EXPERIMENTS:
    for cache in CACHE_SETTINGS.values():
        variant = "cache" if cache else "no-cache"
        for dataset in DATASETS:
            summary_path = (
                ROOT
                / folder
                / variant
                / f"{dataset}_{'cache_' if cache else ''}{box}_box_{task}_budget_results"
                / "budget_summary.csv"
            )
            if summary_path.exists():
                frames.append(pd.read_csv(summary_path))
            else:
                missing.append(str(summary_path))

if missing:
    print("Missing summaries (run those notebooks first):")
    print("\n".join(missing))

results = pd.concat(frames, ignore_index=True)
# 100% keeps every query, so both template rules hold the same run; keep one.
results = results.loc[(results["budget_pct"] == 100) & (results["template_rule"] == "random")]
print(f"{len(results)} runs collected")

48 runs collected


## 2. Build the table

Rows: cache setting → dataset → model. The difference column is 100 × (max − min Macro-F1) over the
three models in the same experiment, cache setting and dataset.

In [3]:
def build_table() -> pd.DataFrame:
    blocks = []
    for label, _folder, task, _box in EXPERIMENTS:
        block = (
            results.loc[results["experiment"] == label]
            .assign(cache=lambda frame: frame["cache"].map(CACHE_LABELS))
            .set_index(["cache", "dataset", "model"])
            [["macro_f1", "gain_over_random_pct", "ow_accuracy"]]
            .rename(
                columns={
                    "macro_f1": "Macro-F1",
                    "gain_over_random_pct": "Macro-F1 gain over random (%)",
                    "ow_accuracy": "OWAcc",
                }
            )
        )
        if task == "topic":
            block = block.drop(columns="OWAcc")
        spread = block.groupby(level=["cache", "dataset"])["Macro-F1"]
        block[DIFFERENCE] = 100 * (spread.transform("max") - spread.transform("min"))
        block.columns = pd.MultiIndex.from_product([[label], block.columns])
        blocks.append(block)
    table = pd.concat(blocks, axis=1)
    order = pd.MultiIndex.from_product(
        [
            [CACHE_LABELS[c] for c in CACHE_SETTINGS.values()],
            [DATASET_LABELS[d] for d in DATASETS],
            MODELS,
        ],
        names=["Cache", "D", "Model"],
    )
    return table.reindex(order)


GREEN = "1F5C2E"
RED = "E8A0A0"
ORANGE = "F6A95E"
MAX_DIFFERENCE = 100.0
MAX_GAIN = 350.0


def gain_cell(value: float) -> str:
    if pd.isna(value):
        return "--"
    text = f"{value:.0f}\\%"
    if value < 0:
        return f"\\cellcolor[HTML]{{{RED}}}{text}"
    strength = int(round(min(value, MAX_GAIN) / MAX_GAIN * 100))
    cell = f"\\cellcolor{{tablegreen!{strength}}}"
    return cell + (f"\\textcolor{{white}}{{{text}}}" if strength > 60 else text)


def difference_strength(value: float) -> int:
    """Share of the full orange, from 0 (white) at 0% to 100 at 100%."""
    return int(round(min(max(value, 0.0), MAX_DIFFERENCE) / MAX_DIFFERENCE * 100))


def to_latex(table: pd.DataFrame, caption: str) -> str:
    columns = list(table.columns)
    n_columns = 1 + len(columns)
    header_groups = []
    for label, _folder, task, _box in EXPERIMENTS:
        width = 4 if task == "freq" else 3
        header_groups.append(f"\\multicolumn{{{width}}}{{c}}{{{label}}}")
    sub_header = [
        {
            "Macro-F1": "Macro-F$_1$",
            "Macro-F1 gain over random (%)": "Gain over random",
            "OWAcc": "OWAcc",
            DIFFERENCE: "Greatest $\\Delta$ models",
        }[sub]
        for _label, sub in columns
    ]
    lines = [
        "\\definecolor{tablegreen}{HTML}{" + GREEN + "}",
        "\\definecolor{tableorange}{HTML}{" + ORANGE + "}",
        "\\begin{table}[t]",
        "\\centering",
        "\\footnotesize",
        "\\setlength{\\tabcolsep}{3pt}",
        f"\\caption{{{caption}}}",
        "\\begin{tabular}{l" + "c" * len(columns) + "}",
        "\\toprule",
        " & " + " & ".join(header_groups) + " \\\\",
        "Model & " + " & ".join(sub_header) + " \\\\",
    ]
    for cache_label, dataset in dict.fromkeys(index[:2] for index in table.index):
        lines.append("\\midrule")
        lines.append(
            f"\\multicolumn{{{n_columns}}}{{l}}{{\\textbf{{{cache_label} -- {dataset}}}}} \\\\"
        )
        block = table.loc[(cache_label, dataset)]
        for row_number, (model, row) in enumerate(block.iterrows()):
            cells = []
            for (_label, sub), value in row.items():
                if sub == DIFFERENCE:
                    # One shaded cell spanning the three model rows. The value
                    # sits on the last row with a negative span, so the colour of
                    # the rows below cannot paint over it.
                    text = (
                        f"\\multirow{{-{len(block)}}}{{*}}{{{value:.0f}\\%}}"
                        if row_number == len(block) - 1
                        else ""
                    )
                    cells.append(f"\\cellcolor{{tableorange!{difference_strength(value)}}}" + text)
                elif sub.startswith("Macro-F1 gain"):
                    cells.append(gain_cell(value))
                else:
                    cells.append("--" if pd.isna(value) else f"{value:.2f}")
            lines.append(f"{model} & " + " & ".join(cells) + " \\\\")
    lines += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]
    return "\n".join(lines)


table = build_table()
table.to_csv(OUTPUT_DIRECTORY / "full_queries_table.csv")
(OUTPUT_DIRECTORY / "full_queries_table.tex").write_text(
    to_latex(table, "Attack performance with the full query set (100\\%)"),
    encoding="utf-8",
)
print(f"Tables written to: {OUTPUT_DIRECTORY.resolve()}")

Tables written to: /Users/marinabuitrago/Desktop/experiments-final/classificators/full_queries_tables


## 3. View the table

Gain column coloured as in the budget table (green for gains, red below random). The
difference column is shaded from white (0%) to orange (100%) and shown once per block of three models.

In [4]:
gain_columns = [column for column in table.columns if column[1].startswith("Macro-F1 gain")]
difference_columns = [column for column in table.columns if column[1] == DIFFERENCE]
value_columns = [
    column for column in table.columns if column not in gain_columns + difference_columns
]
GREEN_RGB = tuple(int(GREEN[i : i + 2], 16) for i in (0, 2, 4))
ORANGE_RGB = tuple(int(ORANGE[i : i + 2], 16) for i in (0, 2, 4))


def difference_style(value: float) -> str:
    strength = difference_strength(value) / 100
    red, green, blue = (round(255 + (channel - 255) * strength) for channel in ORANGE_RGB)
    return f"background-color: rgb({red}, {green}, {blue}); color: #0b0b0b; font-weight: bold"


def gain_style(value: float) -> str:
    if pd.isna(value):
        return ""
    if value < 0:
        return f"background-color: #{RED}; color: #0b0b0b"
    strength = min(value, MAX_GAIN) / MAX_GAIN
    red, green, blue = (round(255 + (channel - 255) * strength) for channel in GREEN_RGB)
    text = "white" if strength > 0.6 else "#0b0b0b"
    return f"background-color: rgb({red}, {green}, {blue}); color: {text}"


def difference_text(frame: pd.DataFrame) -> pd.DataFrame:
    """Show the shared difference only on the first model row of each block."""
    first_model = pd.Series(
        frame.index.get_level_values("Model") == MODELS[0], index=frame.index
    )
    return frame[difference_columns].where(first_model, axis=0)


def style_table(frame: pd.DataFrame, caption: str):
    shown = frame.copy()
    shown[difference_columns] = difference_text(frame)
    return (
        shown.style.format("{:.2f}", subset=value_columns, na_rep="--")
        .format("{:.0f}%", subset=gain_columns, na_rep="--")
        .format("{:.0f}%", subset=difference_columns, na_rep="")
        .map(gain_style, subset=gain_columns)
        # Colour every row of a block from the shared value, not the shown text.
        .apply(
            lambda column: [difference_style(value) for value in frame[column.name]],
            subset=difference_columns,
        )
        .set_caption(caption)
        .set_properties(**{"text-align": "center"})
        .set_table_styles(
            [
                {"selector": "th", "props": [("text-align", "center")]},
                {"selector": "caption", "props": [("font-size", "1.3em"), ("font-weight", "bold")]},
            ]
        )
    )


from IPython.display import display

display(style_table(table, "Full query set (100%)"))